# Effectiveness of thrombolysis and thrombectomy in early (6hr) arrivals of non-mild stroke - discharge mRS 0-2

## Selection criteria

* Arrival within 6 hours of known stroke onset time
* Thrombectomy, if given, given within 12 hours
* Thrombolysis, if given, given within 6 hours
* Stroke type = infarction
* No recorded use of perfusion imaging
* Stroke severity > 5 or labelled as LVO

## Outcome of interest

* Discharge with mRS 0-2

## Key features of interest

* Onset to thrombolysis time
* Onset to thrombectomy time

## Adjustment features used:

* Prior disability
* Stroke severity
* Age
* Congestive heart failure
* Hypertension
* Diabetes
* Atrial fibrillation anticoagulant
* Any atrial fibrillation diagnosis (existing or new)
* Stroke team
* Ethnicity

## Methodology steps

* Select patients by above criteria
* Select features by above criteria
* Split data in 5 sets, where 80% of data is used to train a model, and tets results are generated for the remaining 20% (each patient is in a single test set)
* Train 5 models
* For each model get outcome results for the tets set
* For each model, for patients treated with thrombolysis or thrombectomy, get counterfactual outcomes if they had no treatment
* Calculate the difference between actual outcome and (counterfactual) no-treatment outcomes for treated patients
* Analyse resuls for patients who 1) received thrombectomy only, 2) thrombolysis only, 3) thrombolysis or thrombectomy *

*Note - though the analysis is by different groups, the models are fitted to all groups (no treatment, thrombolysis alone, thrombectomy alone, thrombolysis + thrombectomy).

## Load modules

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import statsmodels.api as sm
from sklearn.metrics import roc_auc_score, balanced_accuracy_score
from sklearn.model_selection import StratifiedKFold
from xgboost import XGBClassifier
from xgboost import XGBRegressor

## Define mRS outcome target

In [ ]:
mrs_target = 2 # Model will set y as being less than or equal to this value

## Define adjustment fields

In [ ]:
non_categorical_features = [
    'prior_disability',
    'stroke_severity',
    'age',
    'congestive_heart_failure',
    'hypertension',
    'diabetes',
    'afib_anticoagulant',
    'any_afib_diagnosis',
    'onset_to_thrombolysis_time',
    'onset_to_thrombectomy_time']

categorical_features = [
    'stroke_team',
    'ethnicity',
]

X_fields = non_categorical_features + categorical_features

## Load and filter data

In [ ]:
data = pd.read_csv("../../data/sam3/cleaned_data.csv", low_memory=False)

# Ensure categorical columns are pandas categorical dtype for XGBoost
for col in categorical_features:
    data[col] = data[col].replace("", "Empty")
    data[col] = data[col].astype("category")

# Ensure non-categorical columns are numeric
for col in non_categorical_features:
    data[col] = pd.to_numeric(data[col], errors="coerce")

print(f"Initial data shape: {data.shape}")

# Remove rows with missing discharge disability
data = data.dropna(subset=["discharge_disability"])

# Keep only infarction cases
data = data[data["infarction"] == 1]

# Keep only onset to arrival < 360 mins
data = data[data["onset_to_arrival_time"] < 360]

# Keep only perfusion_imaging_used == 0
data = data[data["perfusion_imaging_used"] == 0]

# Limit to stroke severity > 1 or lvo = 1
data = data[(data["stroke_severity"] > 1) | (data["lvo"] == 1)]

# Remove rows where thrombolysis is after thrombectomy (if both are present)
both_treatments = (
    (data["onset_to_thrombolysis_time"] < 99999)
    & (data["onset_to_thrombectomy_time"] < 99999)
)
thrombolysis_after_thrombectomy = (
    data["onset_to_thrombolysis_time"] > data["onset_to_thrombectomy_time"]
)
data = data[~(both_treatments & thrombolysis_after_thrombectomy)]

# Missing treatment times -> sentinel
data["onset_to_thrombolysis_time"] = data["onset_to_thrombolysis_time"].fillna(99999)
data["onset_to_thrombectomy_time"] = data["onset_to_thrombectomy_time"].fillna(99999)

# Exclude late thrombectomy (> 720) unless sentinel
data = data[
    (data["onset_to_thrombectomy_time"] < 720)
    | (data["onset_to_thrombectomy_time"] == 99999)
]

# Exclude late thrombolysis (> 360) unless sentinel
data = data[
    (data["onset_to_thrombolysis_time"] < 360)
    | (data["onset_to_thrombolysis_time"] == 99999)
]

print(f"Final data shape: {data.shape}")

# Treatment counts
num_thrombectomy = (data["onset_to_thrombectomy_time"] != 99999).sum()
print(f"Number of patients with thrombectomy: {num_thrombectomy}")

num_both = (
    (data["onset_to_thrombolysis_time"] != 99999)
    & (data["onset_to_thrombectomy_time"] != 99999)
).sum()
print(f"Number of patients with both thrombolysis and thrombectomy: {num_both}")

num_only_thrombectomy = (
    (data["onset_to_thrombectomy_time"] != 99999)
    & (data["onset_to_thrombolysis_time"] == 99999)
).sum()
print(f"Number of patients with only thrombectomy: {num_only_thrombectomy}")

num_thrombolysis = (data["onset_to_thrombolysis_time"] != 99999).sum()
print(f"Number of patients with thrombolysis: {num_thrombolysis}")

num_only_thrombolysis = (
    (data["onset_to_thrombolysis_time"] != 99999)
    & (data["onset_to_thrombectomy_time"] == 99999)
).sum()
print(f"Number of patients with only thrombolysis: {num_only_thrombolysis}")

## Create 5 k-fold splits

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

X = data[X_fields]
y = (data['discharge_disability'] <= mrs_target).astype(int)

train_X_sets = []
train_y_sets = []
test_X_sets = []
test_y_sets = []

for train_index, test_index in skf.split(X, y):
    train_X, test_X = X.iloc[train_index], X.iloc[test_index]
    train_y, test_y = y.iloc[train_index], y.iloc[test_index]
    
    train_X_sets.append(train_X)
    train_y_sets.append(train_y)
    test_X_sets.append(test_X)
    test_y_sets.append(test_y)

## Train and test models

In [ ]:
models = []
test_y_pred_probas = []
test_y_preds = []

for split in range(5):
    # Get data for this split
    train_X = train_X_sets[split]
    train_y = train_y_sets[split]
    test_X = test_X_sets[split]
    test_y = test_y_sets[split]
    # Set up model
    pos = train_y.sum()
    neg = len(train_y) - pos
    model = XGBClassifier(
        enable_categorical=True,
        verbosity=0,
        seed=42,
        eval_metric='logloss',
        scale_pos_weight=(neg / pos)
    )
    # Train model
    model.fit(train_X, train_y)
    models.append(model)
    # Evaluate model with AUC and balanced accuracy
    y_pred_proba = model.predict_proba(test_X)[:, 1]
    y_pred = model.predict(test_X)
    auc = roc_auc_score(test_y, y_pred_proba)
    bal_acc = balanced_accuracy_score(test_y, y_pred)
    test_y_pred_probas.append(y_pred_proba)
    test_y_preds.append(y_pred)
    print(f"Split {split + 1}: AUC = {auc:.4f}, Balanced Accuracy = {bal_acc:.4f}")  

## Predict counterfactual outcomes

To mimic no thrombolysis or thrombectomy, set times to 99999.

In [ ]:
counter_factuals = []

for split in range(5):
    # Get model and data
    model = models[split]
    test_X = test_X_sets[split]
    test_y = test_y_sets[split]

    # Get index of patients who received thrombectomy or thrombolysis
    treated_indices = test_X[
        (test_X['onset_to_thrombectomy_time'] < 99999) |
        (test_X['onset_to_thrombolysis_time'] < 99999)].index

    # Get X and y for these patients
    thrombectomy_X = test_X.loc[treated_indices]
    thrombectomy_y = test_y.loc[treated_indices]

    # Get predicted probabilities for these patients
    y_probs = model.predict_proba(thrombectomy_X)[:, 1]
    
    # No thrombolysis scenario: Set onset_to_thrombolysis_time to 99999 for these patients
    no_thrombolysis_X = thrombectomy_X.copy(deep=True)
    no_thrombolysis_X['onset_to_thrombolysis_time'] = 99999
    y_probs_no_thrombolysis = model.predict_proba(no_thrombolysis_X)[:, 1]

    # No thrombectomy scenario: Set onset_to_thrombectomy_time to 99999 for these patients
    no_thrombectomy_X = thrombectomy_X.copy(deep=True)
    no_thrombectomy_X['onset_to_thrombectomy_time'] = 99999
    y_probs_no_thrombectomy = model.predict_proba(no_thrombectomy_X)[:, 1]

    # No thrombolysis and no thrombectomy scenario: Set both times to 99999
    no_both_X = thrombectomy_X.copy(deep=True)
    no_both_X['onset_to_thrombolysis_time'] = 99999
    no_both_X['onset_to_thrombectomy_time'] = 99999
    y_probs_no_both = model.predict_proba(no_both_X)[:, 1]

    # Combine X, y, and store the counterfactuals for this split
    counter_factuals_split = thrombectomy_X.copy(deep=True)
    counter_factuals_split.insert(0, 'patient_index', treated_indices)
    counter_factuals_split['actual_y'] = thrombectomy_y
    counter_factuals_split['y_prob_treated'] = y_probs
    counter_factuals_split['y_prob_no_thrombolysis'] = y_probs_no_thrombolysis
    counter_factuals_split['y_prob_no_thrombectomy'] = y_probs_no_thrombectomy
    counter_factuals_split['y_prob_no_treatment'] = y_probs_no_both
    counter_factuals.append(counter_factuals_split)

# Combine all counterfactuals into a single DataFrame
counter_factuals_df = pd.concat(counter_factuals, ignore_index=True)

In [ ]:
# Define columns for probabilities and compute log-odds
prob_cols = [
    'y_prob_treated',
    'y_prob_no_thrombolysis',
    'y_prob_no_thrombectomy',
    'y_prob_no_treatment',
]

# Convert predicted probabilities to clipped log-odds columns
for col in prob_cols:
    clipped = counter_factuals_df[col].clip(1e-9, 1 - 1e-9)
    log_odds_col = col.replace('y_prob_', 'log_odds_')
    counter_factuals_df[log_odds_col] = np.log(clipped / (1 - clipped))

diff_pairs = {
    'log_odds_diff_treated_no_thrombolysis': 'log_odds_no_thrombolysis',
    'log_odds_diff_treated_no_thrombectomy': 'log_odds_no_thrombectomy',
    'log_odds_diff_treated_no_treatment': 'log_odds_no_treatment',
}

for diff_col, ref_col in diff_pairs.items():
    counter_factuals_df[diff_col] = (
        counter_factuals_df['log_odds_treated'] - counter_factuals_df[ref_col]
    )

Add 60 minute bins of thrombolysis and thrombectomy time.

In [ ]:
# 60-minute bins up to 720, plus sentinel 99999
bin_edges = list(range(0, 361, 60)) + [99999]

# Labels like 0-120, 120-240, ..., 600-720, No treatment
bin_labels = [
    f"{bin_edges[i]}-{bin_edges[i+1]}" for i in range(len(bin_edges) - 2)] + ["No thrombolysis"]
counter_factuals_df['thrombolysis_bin'] = pd.cut(
    counter_factuals_df['onset_to_thrombolysis_time'],
    bins=bin_edges,
    labels=bin_labels,
    right=True,
    include_lowest=True
)

bin_edges = list(range(0, 721, 60)) + [99999]
bin_labels = [
    f"{bin_edges[i]}-{bin_edges[i+1]}" for i in range(len(bin_edges) - 2)] + ["No thrombectomy"]
counter_factuals_df['thrombectomy_bin'] = pd.cut(
    counter_factuals_df['onset_to_thrombectomy_time'],
    bins=bin_edges,
    labels=bin_labels,
    right=True,
    include_lowest=True
)

## Plot heatmap of benefit vs thrombolysis and thrombectomy time

In [ ]:
# Preoduce heatmap of difference to no treatment for thrombectomy and thrombolysis, with bins on x and y axes
heatmap_data = counter_factuals_df.pivot_table(
    index='thrombectomy_bin',
    columns='thrombolysis_bin',
    values='log_odds_diff_treated_no_treatment',
    aggfunc='mean'
)

# Remove any cells where the count is less than 5 to avoid unreliable estimates
heatmap_counts = counter_factuals_df.pivot_table(
    index='thrombectomy_bin',
    columns='thrombolysis_bin',
    values='patient_index',
    aggfunc='count'
)

# Filter heatmap_data to only include cells with counts >= 10
heatmap_data = heatmap_data.where(heatmap_counts >= 10)

# Create annotation labels without changing heatmap_data numeric dtype
heatmap_annot = heatmap_data.apply(
    lambda col: col.map(lambda v: f"{float(v):.2f}" if pd.notna(v) else "")
)

# Add (n=count) to the annotation labels
heatmap_annot = heatmap_annot + heatmap_counts.apply(
    lambda col: col.map(lambda v: f"\n(n={int(v)})" if pd.notna(v) and v >=10 else ""))


# Plot heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(heatmap_data, cmap='bwr_r', center=0, annot=heatmap_annot, fmt='')
plt.title(f'Log Odds Effect of Treatment (mRS <= {mrs_target})')

# Add lines between all bins
for i in range(len(bin_labels)):
    plt.axvline(x=i, color='black', linestyle='--', linewidth=0.5)
    plt.axhline(y=i, color='black', linestyle='--', linewidth=0.5)

# Count the number of bins for thrombolysis and thrombectomy,
# and draw a solid line at the last bin for each axis.
thrombolysis_bin_count = len(counter_factuals_df['thrombolysis_bin'].cat.categories)
thrombectomy_bin_count = len(counter_factuals_df['thrombectomy_bin'].cat.categories)
plt.axvline(x=thrombolysis_bin_count - 1, color='black', linestyle='-', linewidth=1)
plt.axhline(y=thrombectomy_bin_count - 1, color='black', linestyle='-', linewidth=1)

# Add a label to the colorbar
cbar = plt.gcf().axes[-1]
cbar.set_ylabel('Predicted benefit (Log Odds)', rotation=90, labelpad=15)

plt.xlabel('Thrombolysis Time Bin (onset to thrombolysis mins)')
plt.ylabel('Thrombectomy Time Bin (onset to thrombectomy mins)')
plt.savefig(f'./output/thrombolysis_thrombectomy_heatmap_mrs_{mrs_target}.png', dpi=300, bbox_inches='tight')
plt.show()

# Repeat the heatmap but with lod odds convert to odds (exp log odds)
heatmap_data_odds = np.exp(heatmap_data)
heatmap_annot_odds = heatmap_data_odds.apply(
    lambda col: col.map(lambda v: f"{float(v):.2f}" if pd.notna(v) else "")
)
# Add (n=count) to the annotation labels
heatmap_annot_odds = heatmap_annot_odds + heatmap_counts.apply(
    lambda col: col.map(lambda v: f"\n(n={int(v)})" if pd.notna(v) and v >=10 else "")
)

# Plot heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(heatmap_data_odds, cmap='bwr_r', center=1, annot=heatmap_annot_odds, fmt='')
plt.title(f'Odds Effect of Treatment (mRS <= {mrs_target})')

# Add lines between all bins
for i in range(len(bin_labels)):
    plt.axvline(x=i, color='black', linestyle='--', linewidth=0.5)
    plt.axhline(y=i, color='black', linestyle='--', linewidth=0.5)

# Count the number of bins for thrombolysis and thrombectomy,
# and draw a solid line at the last bin for each axis.
plt.axvline(x=thrombolysis_bin_count - 1, color='black', linestyle='-', linewidth=1)
plt.axhline(y=thrombectomy_bin_count - 1, color='black', linestyle='-', linewidth=1)

# Add a label to the colorbar
cbar = plt.gcf().axes[-1]
cbar.set_ylabel('Predicted benefit (Odds)', rotation=90, labelpad=15)

plt.xlabel('Thrombolysis Time Bin (onset to thrombolysis mins)')
plt.ylabel('Thrombectomy Time Bin (onset to thrombectomy mins)')
plt.savefig(f'./output/thrombolysis_thrombectomy_heatmap_odds_mrs_{mrs_target}.png', dpi=300, bbox_inches='tight')
plt.show()


## Analysis of thrombectomy alone

In [ ]:
# Get data for patients who received only thrombectomy
only_thrombectomy = counter_factuals_df[
    (counter_factuals_df["onset_to_thrombectomy_time"] < 99999)
    & (counter_factuals_df["onset_to_thrombolysis_time"] == 99999)
].copy()

# Fit regression model for predicted benefit vs thrombectomy time
reg_X = only_thrombectomy[["onset_to_thrombectomy_time"]]
reg_y = only_thrombectomy["log_odds_diff_treated_no_treatment"]

reg_X = sm.add_constant(reg_X)
model = sm.OLS(reg_y, reg_X).fit()

print(model.summary())

# Intercept at x = 0 with 95% CI
intercept = model.params["const"]
conf_int = model.conf_int().loc["const"]
print(
    f"\nMax effect (intercept at x=0): {intercept:.4f}, "
    f"95% CI: [{conf_int[0]:.4f}, {conf_int[1]:.4f}]"
)

# Time at which predicted effect is zero: 0 = intercept + slope * x
slope = model.params["onset_to_thrombectomy_time"]

if np.isclose(slope, 0):
    print("Slope is approximately zero, so x at y=0 cannot be estimated reliably.")
else:
    x_at_y0 = -intercept / slope

    cov = model.cov_params()
    var_intercept = cov.loc["const", "const"]
    var_slope = cov.loc[
        "onset_to_thrombectomy_time", "onset_to_thrombectomy_time"
    ]
    cov_intercept_slope = cov.loc["const", "onset_to_thrombectomy_time"]

    var_x_at_y0 = (
        (1 / slope**2) * var_intercept
        + (intercept**2 / slope**4) * var_slope
        - (2 * intercept / slope**3) * cov_intercept_slope
    )
    var_x_at_y0 = max(var_x_at_y0, 0)
    std_x_at_y0 = np.sqrt(var_x_at_y0)

    conf_int_x_at_y0 = (
        x_at_y0 - 1.96 * std_x_at_y0,
        x_at_y0 + 1.96 * std_x_at_y0,
    )
    print(
        f"Time at which effect is zero (x at y=0): {x_at_y0:.0f} mins, "
        f"95% CI: [{conf_int_x_at_y0[0]:.0f}, {conf_int_x_at_y0[1]:.0f}] mins"
    )

# Create a seaborn scatter plot with regression line and 95% CI
plt.figure(figsize=(7, 6))
sns.regplot(
    x="onset_to_thrombectomy_time",
    y="log_odds_diff_treated_no_treatment",
    data=only_thrombectomy,
    ci=95,
    scatter_kws={"s": 10, "alpha": 0.5},
    line_kws={"color": "red"},
)
# Add a line at y=0
plt.axhline(y=0, color='black', linestyle='--', alpha=0.5)
plt.title(
    f"Effect of Thrombectomy Time on Predicted Benefit (mRS <= {mrs_target})\n"
    "(Thrombectomy alone, no thrombolysis)"
)
plt.xlabel('Onset to Thrombectomy Time (mins)')
plt.ylabel(f'Predicted Benefit\n(Log Odds improvement in mRS <= {mrs_target} vs no treatment)')
plt.savefig(
    f'./output/thrombectomy_alone_time_effect_mrs_{mrs_target}.png', dpi=300, bbox_inches='tight')
plt.show()

# Use models[0] to get SHAP values patients receiving only thrombectomy
only_thrombectomy_X = only_thrombectomy[X_fields]
explainer = shap.Explainer(models[0])
shap_values = explainer(only_thrombectomy_X)
# Plot beeswarm plot for SHAP values
shap.plots.beeswarm(shap_values, max_display=99, show=False)
plt.title(f"SHAP Values for Thrombectomy Alone Model (mRS <= {mrs_target})")
plt.savefig(f'./output/shap_beeswarm_thrombectomy_alone_mrs_{mrs_target}.png', dpi=300, bbox_inches='tight')
plt.show()

## Analysis of thrombolysis alone

In [ ]:
# Get data for patients who received only thrombolysis
only_thrombolysis = counter_factuals_df[
    (counter_factuals_df["onset_to_thrombolysis_time"] < 99999)
    & (counter_factuals_df["onset_to_thrombectomy_time"] == 99999)
].copy()

# Fit regression model for predicted benefit vs thrombolysis time
reg_X = only_thrombolysis[["onset_to_thrombolysis_time"]]
reg_y = only_thrombolysis["log_odds_diff_treated_no_treatment"]

reg_X = sm.add_constant(reg_X)
model = sm.OLS(reg_y, reg_X).fit()

print(model.summary())

# Intercept at x = 0 with 95% CI
intercept = model.params["const"]
conf_int = model.conf_int().loc["const"]
print(
    f"\nMax effect (intercept at x=0): {intercept:.4f}, "
    f"95% CI: [{conf_int[0]:.4f}, {conf_int[1]:.4f}]"
)

# Time at which predicted effect is zero: 0 = intercept + slope * x
slope = model.params["onset_to_thrombolysis_time"]

if np.isclose(slope, 0):
    print("Slope is approximately zero, so x at y=0 cannot be estimated reliably.")
else:
    x_at_y0 = -intercept / slope

    cov = model.cov_params()
    var_intercept = cov.loc["const", "const"]
    var_slope = cov.loc[
        "onset_to_thrombolysis_time", "onset_to_thrombolysis_time"
    ]
    cov_intercept_slope = cov.loc["const", "onset_to_thrombolysis_time"]

    var_x_at_y0 = (
        (1 / slope**2) * var_intercept
        + (intercept**2 / slope**4) * var_slope
        - (2 * intercept / slope**3) * cov_intercept_slope
    )
    var_x_at_y0 = max(var_x_at_y0, 0)
    std_x_at_y0 = np.sqrt(var_x_at_y0)

    conf_int_x_at_y0 = (
        x_at_y0 - 1.96 * std_x_at_y0,
        x_at_y0 + 1.96 * std_x_at_y0,
    )
    print(
        f"Time at which effect is zero (x at y=0): {x_at_y0:.0f} mins, "
        f"95% CI: [{conf_int_x_at_y0[0]:.0f}, {conf_int_x_at_y0[1]:.0f}] mins"
    )

# Create a seaborn scatter plot with regression line and 95% CI
plt.figure(figsize=(7, 6))
sns.regplot(
    x="onset_to_thrombolysis_time",
    y="log_odds_diff_treated_no_treatment",
    data=only_thrombolysis,
    ci=95,
    scatter_kws={"s": 10, "alpha": 0.5},
    line_kws={"color": "red"},
)

# Add a line at y=0
plt.axhline(y=0, color="black", linestyle="--", alpha=0.5)

plt.title(
    f"Effect of Thrombolysis Time on Predicted Benefit (mRS <= {mrs_target})\n"
    "(Thrombolysis alone, no thrombectomy)"
)
plt.xlabel("Onset to Thrombolysis Time (mins)")
plt.ylabel(
    f"Predicted Benefit\n(Log Odds improvement in mRS <= {mrs_target} vs no treatment)"
)
plt.savefig(
    f"./output/thrombolysis_alone_time_effect_mrs_{mrs_target}.png", dpi=300, bbox_inches="tight")

plt.show()

# Use models[0] to get SHAP values for patients receiving only thrombolysis
only_thrombolysis_X = only_thrombolysis[X_fields]
explainer = shap.Explainer(models[0])
shap_values = explainer(only_thrombolysis_X)

# Plot beeswarm plot for SHAP values
shap.plots.beeswarm(shap_values, max_display=99, show=False)
plt.title(f"SHAP Values for Thrombolysis Alone Model (mRS <= {mrs_target})")
plt.savefig(
    f"./output/shap_beeswarm_thrombolysis_alone_mrs_{mrs_target}.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

## Analysis of thrombectomy and thrombolysis (analyse time to thrombectomy)

In [ ]:
# Get data for patients who thrombectomy and thrombolysis
both_treatments = counter_factuals_df[
    (counter_factuals_df["onset_to_thrombectomy_time"] < 99999)
    & (counter_factuals_df["onset_to_thrombolysis_time"] < 99999)
].copy()

# Fit regression model for predicted benefit vs thrombectomy time
reg_X = both_treatments[["onset_to_thrombectomy_time"]]
reg_y = both_treatments["log_odds_diff_treated_no_treatment"]

reg_X = sm.add_constant(reg_X)
model = sm.OLS(reg_y, reg_X).fit()

print(model.summary())

# Intercept at x = 0 with 95% CI
intercept = model.params["const"]
conf_int = model.conf_int().loc["const"]
print(
    f"\nMax effect (intercept at x=0): {intercept:.4f}, "
    f"95% CI: [{conf_int[0]:.4f}, {conf_int[1]:.4f}]"
)

# Time at which predicted effect is zero: 0 = intercept + slope * x
slope = model.params["onset_to_thrombectomy_time"]

if np.isclose(slope, 0):
    print("Slope is approximately zero, so x at y=0 cannot be estimated reliably.")
else:
    x_at_y0 = -intercept / slope

    cov = model.cov_params()
    var_intercept = cov.loc["const", "const"]
    var_slope = cov.loc[
        "onset_to_thrombectomy_time", "onset_to_thrombectomy_time"
    ]
    cov_intercept_slope = cov.loc["const", "onset_to_thrombectomy_time"]

    var_x_at_y0 = (
        (1 / slope**2) * var_intercept
        + (intercept**2 / slope**4) * var_slope
        - (2 * intercept / slope**3) * cov_intercept_slope
    )
    var_x_at_y0 = max(var_x_at_y0, 0)
    std_x_at_y0 = np.sqrt(var_x_at_y0)

    conf_int_x_at_y0 = (
        x_at_y0 - 1.96 * std_x_at_y0,
        x_at_y0 + 1.96 * std_x_at_y0,
    )
    print(
        f"Time at which effect is zero (x at y=0): {x_at_y0:.0f} mins, "
        f"95% CI: [{conf_int_x_at_y0[0]:.0f}, {conf_int_x_at_y0[1]:.0f}] mins"
    )

# Create a seaborn scatter plot with regression line and 95% CI
plt.figure(figsize=(7, 6))
sns.regplot(
    x="onset_to_thrombectomy_time",
    y="log_odds_diff_treated_no_treatment",
    data=both_treatments,
    ci=95,
    scatter_kws={"s": 10, "alpha": 0.5},
    line_kws={"color": "red"},
)
# Add a line at y=0
plt.axhline(y=0, color='black', linestyle='--', alpha=0.5)
plt.title(
    f"Effect of Thrombectomy Time on Predicted Benefit (mRS <= {mrs_target})\n"
    "(Thrombectomy and thrombolysis)"
)
plt.xlabel('Onset to Thrombectomy Time (mins)')
plt.ylabel(f'Predicted Benefit\n(Log Odds improvement in mRS <= {mrs_target} vs no treatment)')
plt.savefig(f'./output/thrombectomy_and_thrombolysis_thrombectomy_time_effect_mrs_{
    mrs_target}.png', dpi=300, bbox_inches='tight')
plt.show()

# Use patients who received both thrombolysis and thrombectomy
both_treatments_X = both_treatments[X_fields]
explainer = shap.Explainer(models[0])
shap_values = explainer(both_treatments_X)

# Plot beeswarm plot for SHAP values
shap.plots.beeswarm(shap_values, max_display=99, show=False)
plt.title(f"SHAP Values for Both Treatments Model (mRS <= {mrs_target})")
plt.savefig(
    f"./output/shap_beeswarm_both_treatments_mrs_{mrs_target}.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()


## Analysis of thrombectomy and thrombolysis (analyse time to thrombolysis)

In [ ]:
# Fit regression model for predicted benefit vs thrombolysis time
reg_X = both_treatments[["onset_to_thrombolysis_time"]]
reg_y = both_treatments["log_odds_diff_treated_no_treatment"]

reg_X = sm.add_constant(reg_X)
model = sm.OLS(reg_y, reg_X).fit()

print(model.summary())

# Intercept at x = 0 with 95% CI
intercept = model.params["const"]
conf_int = model.conf_int().loc["const"]
print(
    f"\nMax effect (intercept at x=0): {intercept:.4f}, "
    f"95% CI: [{conf_int[0]:.4f}, {conf_int[1]:.4f}]"
)

# Time at which predicted effect is zero: 0 = intercept + slope * x
slope = model.params["onset_to_thrombolysis_time"]

if np.isclose(slope, 0):
    print("Slope is approximately zero, so x at y=0 cannot be estimated reliably.")
else:
    x_at_y0 = -intercept / slope

    cov = model.cov_params()
    var_intercept = cov.loc["const", "const"]
    var_slope = cov.loc[
        "onset_to_thrombolysis_time", "onset_to_thrombolysis_time"
    ]
    cov_intercept_slope = cov.loc["const", "onset_to_thrombolysis_time"]

    var_x_at_y0 = (
        (1 / slope**2) * var_intercept
        + (intercept**2 / slope**4) * var_slope
        - (2 * intercept / slope**3) * cov_intercept_slope
    )
    var_x_at_y0 = max(var_x_at_y0, 0)
    std_x_at_y0 = np.sqrt(var_x_at_y0)

    conf_int_x_at_y0 = (
        x_at_y0 - 1.96 * std_x_at_y0,
        x_at_y0 + 1.96 * std_x_at_y0,
    )
    print(
        f"Time at which effect is zero (x at y=0): {x_at_y0:.0f} mins, "
        f"95% CI: [{conf_int_x_at_y0[0]:.0f}, {conf_int_x_at_y0[1]:.0f}] mins"
    )

# Create a seaborn scatter plot with regression line and 95% CI
plt.figure(figsize=(7, 6))
sns.regplot(
    x="onset_to_thrombolysis_time",
    y="log_odds_diff_treated_no_treatment",
    data=both_treatments,
    ci=95,
    scatter_kws={"s": 10, "alpha": 0.5},
    line_kws={"color": "red"},
)
# Add a line at y=0
plt.axhline(y=0, color='black', linestyle='--', alpha=0.5)
plt.title(
    f"Effect of Thrombolysis Time on Predicted Benefit (mRS <= {mrs_target})\n"
    "(Thrombectomy and thrombolysis)"
)
plt.xlabel('Onset to Thrombolysis Time (mins)')
plt.ylabel(f'Predicted Benefit\n(Log Odds improvement in mRS <= {mrs_target} vs no treatment)')
plt.savefig(f'./output/thrombectomy_and_thrombolysis_thrombolysis_time_effect_mrs_{
    mrs_target}.png', dpi=300, bbox_inches='tight')
plt.show()


SHAP plot is the same as above.

## Additional analysis

### Relationship between times to thrombolysis and thrombectomy

In [ ]:
# When both thrombolysis and thrombecrtomy used, fit a SMS regression between those times
reg_X = both_treatments[["onset_to_thrombolysis_time"]]
reg_y = both_treatments[["onset_to_thrombectomy_time"]]
reg_X = sm.add_constant(reg_X)
model = sm.OLS(reg_y, reg_X).fit()
print(model.summary())

# Create a seaborn scatter plot with regression line and 95% CI
plt.figure(figsize=(7, 6))
sns.regplot(
    x="onset_to_thrombolysis_time",
    y="onset_to_thrombectomy_time",
    data=both_treatments,
    ci=95,
    scatter_kws={"s": 10, "alpha": 0.5},
    line_kws={"color": "red"},
)
# Add a line at y=0
plt.axhline(y=0, color='black', linestyle='--', alpha=0.5)
plt.title(
    f"Relationship between Thrombolysis and Thrombectomy Times"
)
plt.xlabel('Onset to Thrombolysis Time (mins)')
plt.ylabel(f'Onset to Thrombectomy Time (mins)')        
plt.savefig(f'./output/thrombectomy_and_thrombolysis_thrombolysis_times.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
diff_in_times = both_treatments["onset_to_thrombectomy_time"] - both_treatments["onset_to_thrombolysis_time"]
diff_in_times.describe().round(0)

In [ ]:
# Create histogram of difference in times between thrombectomy and thrombolysis
# Create 15 min bins up to maximum difference
max_diff = diff_in_times.max()
bins = list(range(0, int(max_diff) + 1, 15))

plt.figure(figsize=(7, 6))
plt.hist(diff_in_times, bins=bins, edgecolor='black', alpha=0.7)
plt.xlabel('Difference in Times (mins)')
plt.ylabel('Frequency')
# Make scale in 60 mins
plt.xticks(range(0, 601, 60))
plt.title('Distribution of Time Differences between Thrombectomy and Thrombolysis')
plt.savefig(f'./output/difference_in_thrombectomy_and_thrombolysis_times_hist.png',
            dpi=300, bbox_inches='tight')
plt.show()